# 사진 : 누락·분리·정션·추가 잉크 구조 점수

설문·추가 학습 없이 누락/분리/정션과 **기준으로 설명되지 않는 추가 잉크**를 감점합니다.
높을수록 측정 오류가 적습니다. 사람 선호나 글자 품질에 교정한 점수는 아닙니다.

1. 첫 셀에서 입력과 가중치/변환 크기를 설정합니다.
2. 모듈 로드 셀과 계산 셀을 실행합니다.
3. 사진 아래의 **전체 점수**를 비교합니다. 카드의 **획별 점수 보기**를 누르면 세 오류와 획별 점수가 펼쳐집니다.

저장된 NPZ의 실제 평가 입력 배열을 우선 표시하고, 없으면 원본 PNG 경로에서 사진을 읽습니다.
기준 획 이미지나 진단 몽타주를 원본 대신 표시하지 않습니다. 경로가 없으면 이미지 없음으로 알립니다.
사진과 점수를 함께 담은 HTML/PNG도 결과 폴더에 저장합니다.

기존 구조 진단 노트북에 추가 셀을 붙이면 structure_results를 바로 재사용합니다.
이 독립 파일을 새로 열었다면 REPORT_PATH를 결과 JSON 경로로 지정하세요.
MODE="values"는 숫자만 입력하는 테스트입니다. 기존 추출 모델을 재실행하지 않습니다.

기본 모드는 **획 추가·삭제를 강하게, 정션을 중간, 분리를 약하게** 감점하는 평가 규칙입니다.
분리 유지율 = 1 - 분리최대감점 × (wD·D/sD)/(1+wD·D/sD)
정션 유지율 = 1 - 정션최대감점 × (wJ·J/sJ)/(1+wJ·J/sJ)
획 점수 = 100 × (1-M)^wM × 분리 유지율 × 정션 유지율
평균/최악 획 혼합점수와 각 오류별 점수 상한 중 최솟값을 사용합니다.
삭제 상한 = 최악 획의 누락 유지율 × 100. 추가 상한 = 100 × exp(-wE·max(E-허용량,0)/추가scale).
단독 분리/정션은 기본값에서 최대 60/65점 감점이고, 복합 오류는 더 낮아질 수 있습니다.
카드의 '오류별 점수 상한 보기'에서 점수가 낮아진 원인을 확인하세요.

기본값에서 누락 1인 획은 0점, 세 오류가 0인 획은 100점입니다.
각 카드의 '추가 잉크 위치 보기'에서 빨간 영역을 확인하세요. 추가는 획 배정과 무관하게 입력 전체에서 측정합니다.
기준 위치는 제한된 범위에서 부드럽게 정렬합니다. 큰 변형이나 정렬 실패는 추가로 오검출할 수 있습니다.
반경과 허용량은 정상 자료에서 교정해야 합니다. 이름별로 점수나 오류 순서를 강제하지 않습니다.
오래된 JSON도 NPZ의 image/reference_geometry가 있으면 추가를 재측정합니다. 없으면 0으로 가정하지 않고 오류를 냅니다.


In [ ]:
# ===== 점수 설정: 누락·분리·정션 + 추가 잉크 =====
# auto: 현재 structure_results가 있으면 재사용, 없으면 REPORT_PATH 읽기
# values: 아래 VALUES의 숫자만으로 바로 계산
MODE = "auto"
REPORT_PATH = "/content/drive/Shareddrives/2026 자율연구/HSQM/stroke_zip_runs/20261028_082616_066020/structure_20261101_092625_060835/structure_report.json"
SAVE_DIR = None  # None = 원래 결과 폴더 아래 indicator_시간. 직접 지정 시 새 폴더
SHOW_IMAGES = True  # 사진과 점수를 카드로 함께 표시
IMAGES_PER_ROW = 3
IMAGES_PER_PAGE = 12  # 저장용 PNG 한 장에 들어갈 사진 수
SHOW_STROKE_TABLE = False  # 기본은 카드의 '획별 점수 보기'를 눌러 확인

SCORE_CONFIG = dict(
    missing_weight=1.0,   # 누락 중요도. 0이면 이 항 제외
    split_weight=1.0,     # 분리 중요도
    junction_weight=1.0,  # 정션 중요도
    split_scale=0.2,      # 기본 모드: 이 크기에서 분리 최대 감점의 절반
    junction_scale=0.01,  # 기본 모드: 이 크기에서 정션 최대 감점의 절반
    worst_stroke_weight=0.5, # 글자 점수: 획 평균과 최악 획 점수의 혼합. 0=평균, 1=최악
    priority_mode=True,   # 삭제/추가 강한 감점, 정션/분리 제한 감점. False=이전 공식
    split_max_penalty=0.60, # 분리 단독 최대 감점 60점
    junction_max_penalty=0.65, # 정션 단독 최대 감점 65점
    extra_weight=1.0,     # 추가 잉크 감점. 0은 명시적으로 추가 검사를 끄는 설정
    extra_scale=0.025,    # 허용량 초과 2.5%p일 때 추가 점수 상한 100/e
    extra_allowance=0.03, # 정렬 후 남은 작은 잉크 차이 3%까지 감점 유예
    extra_radius=8.0,     # 목표 기준 중심선 주변의 허용 반경 (128 해상도 픽셀)
    extra_transition=0.5, # 허용 영역 경계의 부드러운 전이 폭
    extra_alignment=6,    # 기준 위치를 ±6px 안에서만 이동 허용
    extra_alignment_step=2, # 이동 후보 간격
    extra_alignment_temperature=0.01, # 부드러운 위치 선택 온도
    extra_alignment_prior=0.001, # 큰 이동을 억제하는 비용
    extra_alignment_sigma=2.0, # 위치 일치도를 계산하는 중심선 주변 폭
    eps=1e-6,
)

# MODE="values"일 때 사용. 이미지나 모델 없이 이 숫자만 계산합니다.
VALUES = [
    dict(stroke=1, missing=0.0, split=0.0, junction=0.0),
    dict(stroke=2, missing=0.809, split=0.0, junction=0.0),
]
EXTRA_VALUE = 0.0  # MODE="values": 글자 전체의 추가 잉크 비율을 직접 지정


In [ ]:
# 점수 모듈 내장: 추가 모델·가중치 다운로드 없이 실행
import sys, types
_extra_module = types.ModuleType("_extra_ink_runtime")
_extra_module.__file__ = "<embedded:extra_ink>"
sys.modules[_extra_module.__name__] = _extra_module
exec(compile('"""Reference-aware unexplained ink, measured on the INPUT, not assigned masks.\n\nReference centerlines define tolerant support regions with bounded translations. This\ndetects both attached and detached added ink even if every input pixel was\nassigned to an existing stroke by softmax. It is not an extra-stroke counter.\n"""\nimport math\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt, shift\nfrom skimage.morphology import skeletonize\nimport torch\nfrom torch import nn\n\n\nclass ExtraInkMetric(nn.Module):\n    def __init__(self, aligned_reference_strokes, radius=8., transition=0.5,\n                 min_ink_mass=1., reference_threshold=0.5, alignment=6,\n                 alignment_step=2, alignment_temperature=.01, alignment_prior=.001,\n                 alignment_sigma=2.):\n        super().__init__()\n        if not all(math.isfinite(x) for x in (radius, transition, min_ink_mass, reference_threshold)):\n            raise ValueError(\'Extra ink settings must be finite\')\n        if radius < 0 or transition <= 0 or min_ink_mass <= 0 or not 0 < reference_threshold < 1:\n            raise ValueError(\'Invalid extra ink radius, transition, mass or threshold\')\n        if (not isinstance(alignment, int) or alignment < 0 or not isinstance(alignment_step, int)\n                or alignment_step < 1 or not math.isfinite(alignment_temperature)\n                or alignment_temperature <= 0 or not math.isfinite(alignment_prior) or alignment_prior < 0\n                or not math.isfinite(alignment_sigma) or alignment_sigma <= 0):\n            raise ValueError(\'Invalid bounded alignment settings\')\n        reference = torch.as_tensor(aligned_reference_strokes).detach().float().cpu().numpy()\n        if reference.ndim != 3 or not np.isfinite(reference).all() or (reference < 0).any() or (reference > 1).any():\n            raise ValueError(\'Reference must be finite K,H,W ink in [0,1]\')\n        if not len(reference) or any(not (s > reference_threshold).any() for s in reference):\n            raise ValueError(\'Keep every expected reference stroke; empty reference strokes are invalid\')\n        # Fixed translation banks; prediction-dependent weights stay in torch.\n        offsets = sorted(set(range(-alignment, alignment+1, alignment_step)) | {0, alignment})\n        coords = [(dy, dx) for dy in offsets for dx in offsets]\n        matches, supports = [], []\n        for stroke in reference:\n            distance = distance_transform_edt(~skeletonize(stroke > reference_threshold))\n            match = np.exp(-.5*(distance/alignment_sigma)**2)\n            support = 1/(1+np.exp(np.clip((distance-radius)/transition, -80, 80)))\n            match_bank, support_bank = [], []\n            for dy, dx in coords:\n                moved = shift(match, (dy, dx), order=0, mode=\'constant\')\n                match_bank.append(moved/max(moved.sum(), 1e-8))\n                support_bank.append(shift(support, (dy, dx), order=0, mode=\'constant\'))\n            matches.append(match_bank)\n            supports.append(support_bank)\n        self.register_buffer(\'matches\', torch.tensor(np.asarray(matches), dtype=torch.float32))\n        self.register_buffer(\'supports\', torch.tensor(np.asarray(supports), dtype=torch.float32))\n        self.register_buffer(\'shift_cost\', torch.tensor([alignment_prior*(dy*dy+dx*dx)\n                                                       for dy, dx in coords], dtype=torch.float32))\n        self.alignment_temperature = alignment_temperature\n        self.min_ink_mass = min_ink_mass\n\n    def forward(self, image):\n        image = torch.as_tensor(image)\n        if image.ndim == 2:\n            image = image[None, None]\n        elif image.ndim == 3:\n            image = image[None]\n        if image.ndim != 4 or image.shape[1] != 1 or image.shape[-2:] != self.supports.shape[-2:]:\n            raise ValueError(\'image must have shape H,W or 1,H,W or B,1,H,W matching the reference\')\n        if not image.is_floating_point():\n            image = image.float()\n        if image.device != self.supports.device:\n            raise ValueError(\'Move ExtraInkMetric to the image device\')\n        if not bool(torch.isfinite(image).all()) or bool((image < 0).any()) or bool((image > 1).any()):\n            raise ValueError(\'Image must be finite absolute ink in [0,1]\')\n        matches = self.matches.to(dtype=image.dtype)\n        supports = self.supports.to(dtype=image.dtype)\n        alignment_scores = torch.einsum(\'bchw,knhw->bkn\', image, matches)-self.shift_cost\n        weights = torch.softmax(alignment_scores/self.alignment_temperature, dim=-1)\n        per_stroke_support = torch.einsum(\'bkn,knhw->bkhw\', weights, supports)\n        support = 1-(1-per_stroke_support).prod(1, keepdim=True)\n        extra_map = image * (1-support)\n        extra_mass = extra_map.flatten(1).sum(1)\n        fraction = extra_mass / image.flatten(1).sum(1).clamp_min(self.min_ink_mass)\n        return {\'extra\': fraction, \'extra_map\': extra_map, \'support\': support, \'alignment_weights\': weights,\n                \'extra_mass\': extra_mass}\n', _extra_module.__file__, "exec"), _extra_module.__dict__)
_score_module = types.ModuleType("_structural_indicator_runtime")
_score_module.__file__ = "<embedded:structural_indicator>"
sys.modules[_score_module.__name__] = _score_module
exec(compile('"""Deterministic structural indicator with explicit added-ink penalty.\n\nNo survey, fitted parameters, image embeddings, width or hard branch counts.\nThe 0..100 scale is a configurable engineering index, not calibrated quality.\n"""\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime\nimport json\nimport math\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom _extra_ink_runtime import ExtraInkMetric\n\n\n@dataclass(frozen=True)\nclass IndicatorConfig:\n    missing_weight: float = 1.0\n    split_weight: float = 1.0\n    junction_weight: float = 1.0\n    split_scale: float = 0.2\n    junction_scale: float = 0.01\n    worst_stroke_weight: float = 0.5\n    priority_mode: bool = True\n    split_max_penalty: float = 0.60\n    junction_max_penalty: float = 0.65\n    extra_weight: float = 1.0\n    extra_scale: float = 0.025\n    extra_allowance: float = 0.03\n    extra_radius: float = 8.0\n    extra_transition: float = 0.5\n    extra_alignment: int = 6\n    extra_alignment_step: int = 2\n    extra_alignment_temperature: float = 0.01\n    extra_alignment_prior: float = 0.001\n    extra_alignment_sigma: float = 2.0\n    eps: float = 1e-6\n\n    def __post_init__(self):\n        for name, value in asdict(self).items():\n            if not math.isfinite(value):\n                raise ValueError(f\'{name} must be finite\')\n        if min(self.missing_weight, self.split_weight, self.junction_weight, self.extra_weight) < 0:\n            raise ValueError(\'Weights must be nonnegative\')\n        if self.split_scale <= 0 or self.junction_scale <= 0 or not 0 < self.eps < 1:\n            raise ValueError(\'Scales must be positive; eps must be in (0,1)\')\n        if self.extra_scale <= 0 or not 0 <= self.extra_allowance < 1 or self.extra_radius < 0 or self.extra_transition <= 0:\n            raise ValueError(\'Invalid extra-ink scale, allowance or geometry settings\')\n        for weight, scale in ((self.split_weight, self.split_scale), (self.junction_weight, self.junction_scale), (self.extra_weight, self.extra_scale)):\n            if not math.isfinite(weight/scale) or weight/scale > torch.finfo(torch.float32).max:\n                raise ValueError(\'weight/scale must fit a finite float32 coefficient\')\n        if not 0 <= self.worst_stroke_weight <= 1:\n            raise ValueError(\'worst_stroke_weight must be in [0,1]\')\n        if not isinstance(self.priority_mode, bool):\n            raise ValueError(\'priority_mode must be bool\')\n        if not 0 <= self.split_max_penalty <= self.junction_max_penalty < 1:\n            raise ValueError(\'Require 0 <= split_max_penalty <= junction_max_penalty < 1\')\n        if (not isinstance(self.extra_alignment, int) or self.extra_alignment < 0\n                or not isinstance(self.extra_alignment_step, int) or self.extra_alignment_step < 1\n                or self.extra_alignment_temperature <= 0 or self.extra_alignment_prior < 0\n                or self.extra_alignment_sigma <= 0):\n            raise ValueError(\'Invalid bounded alignment settings\')\n\n\nclass StructuralIndicator(nn.Module):\n    """Reference error scores with configurable severity priorities.\n\n    Priority mode bounds isolated split/junction penalties and caps the glyph\n    score by its worst missing stroke and unexplained ink. Combined defects\n    can still score below the isolated-error bands. min uses subgradients.\n    priority_mode=False reproduces the legacy exponential aggregation.\n    """\n    def __init__(self, config=None):\n        super().__init__()\n        self.config = config or IndicatorConfig()\n\n    def forward(self, missing, split, junction, active=None, *, extra=None):\n        missing = torch.as_tensor(missing)\n        if not missing.is_floating_point():\n            missing = missing.float()\n        split = torch.as_tensor(split, device=missing.device, dtype=missing.dtype)\n        junction = torch.as_tensor(junction, device=missing.device, dtype=missing.dtype)\n        if missing.ndim not in (1, 2) or missing.shape != split.shape or missing.shape != junction.shape:\n            raise ValueError(\'missing, split, junction must have matching K or B,K shapes\')\n        if active is None:\n            active = torch.ones_like(missing, dtype=torch.bool)\n        if active.dtype != torch.bool or active.shape != missing.shape or active.device != missing.device:\n            raise ValueError(\'active must be a bool tensor matching the input shape/device\')\n        if missing.ndim == 1:\n            missing, split, junction, active = [x[None] for x in (missing, split, junction, active)]\n        for name, value in [(\'missing\', missing), (\'split\', split), (\'junction\', junction)]:\n            if not bool(torch.isfinite(value).all()) or bool((value < 0).any()):\n                raise ValueError(f\'{name} must be finite and nonnegative\')\n        if bool((missing > 1).any()) or not bool(active.any(1).all()):\n            raise ValueError(\'Missing must be <=1 and each sample needs an active stroke\')\n        cfg = self.config\n        # Smooth endpoint extension: exact 0/1 retention with finite gradients,\n        # including fractional weights. Unlike a hard zero branch, full missing\n        # still receives a restorative gradient at the default weight of one.\n        if cfg.missing_weight == 0:\n            retain_m = 1 + missing*0\n        else:\n            epsilon = missing.new_tensor(cfg.eps)\n            lower = epsilon.pow(cfg.missing_weight)\n            denominator = (1+epsilon).pow(cfg.missing_weight)-lower\n            retain_m = ((1-missing+epsilon).pow(cfg.missing_weight)-lower)/denominator\n        retain_d = torch.exp(-(cfg.split_weight/cfg.split_scale) * split)\n        retain_j = torch.exp(-(cfg.junction_weight/cfg.junction_scale) * junction)\n        if cfg.priority_mode:\n            d = (cfg.split_weight/cfg.split_scale)*split\n            j = (cfg.junction_weight/cfg.junction_scale)*junction\n            retain_d = 1-cfg.split_max_penalty*(1-1/(1+d))\n            retain_j = 1-cfg.junction_max_penalty*(1-1/(1+j))\n        stroke_scores = 100 * retain_m * retain_d * retain_j\n        mean = (stroke_scores * active).sum(1) / active.sum(1)\n        worst = stroke_scores.masked_fill(~active, torch.inf).amin(1)\n        base_score = (1-cfg.worst_stroke_weight)*mean + cfg.worst_stroke_weight*worst\n        cap = lambda retain: 100*retain.masked_fill(~active, torch.inf).amin(1)\n        missing_cap, split_cap, junction_cap = [cap(r) for r in (retain_m, retain_d, retain_j)]\n        if cfg.priority_mode:\n            base_score = torch.stack([base_score, missing_cap, split_cap, junction_cap]).amin(0)\n        if extra is None:\n            if cfg.extra_weight > 0:\n                raise ValueError(\'Added ink is required: provide extra or explicitly set extra_weight=0\')\n            extra = base_score*0\n        extra = torch.as_tensor(extra, device=missing.device, dtype=missing.dtype)\n        if extra.ndim == 0 and len(base_score) == 1:\n            extra = extra[None]\n        if extra.shape != base_score.shape or not bool(torch.isfinite(extra).all()) or bool(((extra < 0) | (extra > 1)).any()):\n            raise ValueError(\'extra must contain one finite [0,1] fraction per glyph\')\n        extra_penalty = torch.relu(extra-cfg.extra_allowance)\n        extra_retention = torch.exp(-(cfg.extra_weight/cfg.extra_scale)*extra_penalty)\n        extra_cap = 100*extra_retention\n        score = torch.minimum(base_score, extra_cap) if cfg.priority_mode else base_score * extra_retention\n        return {\'score\': score, \'stroke_scores\': stroke_scores, \'mean_score\': mean,\n                \'worst_score\': worst, \'penalty\': 1-score/100, \'active\': active,\n                \'base_score\': base_score, \'extra\': extra, \'extra_retention\': extra_retention,\n                \'caps\': {\'missing\': missing_cap, \'split\': split_cap, \'junction\': junction_cap, \'extra\': extra_cap},\n                \'retention\': {\'missing\': retain_m, \'split\': retain_d, \'junction\': retain_j}}\n\n\ndef _extra_for_sample(sample, source_path, config):\n    """Old reports are remeasured from snapshots; never assume missing extra=0."""\n    import numpy as np\n    if \'extra\' in sample:\n        return float(sample[\'extra\']), None, \'provided_value\'\n    if config.extra_weight == 0:\n        return 0., None, \'disabled\'\n    for key in (\'npz\', \'input_npz\'):\n        value = sample.get(key)\n        if not value:\n            continue\n        paths = [Path(value)]\n        if source_path and not paths[0].is_absolute():\n            paths.append(source_path.parent/paths[0])\n        for path in paths:\n            if not path.is_file():\n                continue\n            with np.load(path, allow_pickle=False) as arrays:\n                target_key = next((key for key in (\'reference_image\', \'reference\', \'reference_geometry\', \'reference_strokes\')\n                                   if key in arrays), None)\n                if \'image\' not in arrays or target_key not in arrays:\n                    continue\n                image = torch.tensor(arrays[\'image\'], dtype=torch.float32)\n                target = torch.tensor(arrays[target_key], dtype=torch.float32)\n            metric = ExtraInkMetric(target, radius=config.extra_radius, transition=config.extra_transition,\n                alignment=config.extra_alignment, alignment_step=config.extra_alignment_step,\n                alignment_temperature=config.extra_alignment_temperature, alignment_prior=config.extra_alignment_prior,\n                alignment_sigma=config.extra_alignment_sigma)\n            with torch.no_grad():\n                result = metric(image)\n            return float(result[\'extra\'][0]), {\'image\': image.squeeze().numpy(),\n                \'extra_map\': result[\'extra_map\'][0, 0].numpy(),\n                \'support\': result[\'support\'][0, 0].numpy(),\n                \'alignment_weights\': result[\'alignment_weights\'][0].numpy()}, f\'{path}::{target_key}\'\n    raise ValueError(\'Cannot assess added ink: keep the NPZ containing image and reference_geometry/reference_strokes. \'\n                     \'Do not treat absent extra measurements as zero; extra_weight=0 explicitly opts out.\')\n\n\ndef _image_for_sample(sample, source_path):\n    """Prefer the scored input snapshot; never substitute a reference/montage."""\n    import numpy as np\n    from PIL import Image, ImageOps\n\n    def candidates(value):\n        if not value:\n            return []\n        path = Path(value)\n        choices = [path]\n        if not path.is_absolute() and source_path:\n            choices.append(source_path.parent/path)\n        return choices\n\n    for key in (\'npz\', \'input_npz\'):\n        for path in candidates(sample.get(key)):\n            if not path.is_file() or path.suffix.lower() != \'.npz\':\n                continue\n            with np.load(path, allow_pickle=False) as data:\n                if \'image\' not in data:\n                    continue\n                ink = np.asarray(data[\'image\'], dtype=np.float32)\n            if ink.ndim == 3 and ink.shape[0] == 1:\n                ink = ink[0]\n            if ink.ndim != 2 or not np.isfinite(ink).all() or (ink < 0).any() or (ink > 1).any():\n                continue\n            # Saved arrays use ink=1. Display black ink on a white background.\n            content = np.rint(255*(1-ink)).astype(np.uint8)\n            return Image.fromarray(content).convert(\'RGB\'), str(path), \'scored_input_array\'\n    for path in candidates(sample.get(\'input\')):\n        if path.suffix.lower() in (\'.png\', \'.jpg\', \'.jpeg\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\') and path.is_file():\n            with Image.open(path) as image:\n                return ImageOps.exif_transpose(image).convert(\'RGB\'), str(path), \'original_file\'\n    return None, None, \'unavailable\'\n\n\ndef _score_gallery(samples, scored, destination, source_path, columns, per_page, show):\n    import base64\n    import html\n    import io\n    import unicodedata\n    import matplotlib.pyplot as plt\n\n    cards, loaded, pngs = [], [], []\n    for number, (sample, result) in enumerate(zip(samples, scored), 1):\n        image, path, kind = _image_for_sample(sample, source_path)\n        result.update(image_available=image is not None, image_source=path, image_kind=kind)\n        name = str(sample.get(\'input\') or sample.get(\'input_npz\') or f\'Image {number}\').replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1]\n        loaded.append((image, name, result))\n        if image is None:\n            picture = \'<div class="indicator-no-image">사진 경로를 찾지 못했습니다</div>\'\n        else:\n            buffer = io.BytesIO()\n            image.save(buffer, format=\'PNG\')\n            encoded = base64.b64encode(buffer.getvalue()).decode(\'ascii\')\n            picture = f\'<img src="data:image/png;base64,{encoded}" alt="{html.escape(name, quote=True)}">\'\n        detail_rows = \'\'.join(f"<tr><td>{html.escape(str(s[\'stroke\']))}</td><td>{s[\'stroke_score\']:.1f}</td>"\n                              f"<td>{s[\'missing\']:.3g}</td><td>{s[\'split\']:.3g}</td><td>{s[\'junction\']:.3g}</td></tr>"\n                              for s in result[\'strokes\'])\n        note = \'평가에 사용한 입력 배열\' if kind == \'scored_input_array\' else \'입력 사진\'\n        extra_detail = \'\'\n        if result.get(\'extra_overlay\'):\n            encoded = base64.b64encode(Path(result[\'extra_overlay\']).read_bytes()).decode(\'ascii\')\n            extra_detail = (\'<details><summary>추가 잉크 위치 보기</summary>\'\n                f\'<img src="data:image/png;base64,{encoded}" alt="추가 잉크 위치">\'\n                \'<p>빨간색은 기준 허용 영역 밖의 잉크입니다.</p></details>\')\n        extra_note = (f"추가 잉크 {100*result[\'extra\']:.1f}% · 추가 감점 전 {result[\'score_before_extra\']:.1f}점"\n                      if result[\'extra_assessed\'] else \'추가 잉크 검사 꺼짐\')\n        cap_rows = \'\'.join(f"<tr><td>{label}</td><td>{result[key+\'_score_cap\']:.1f}</td></tr>"\n                           for key, label in [(\'missing\', \'누락\'), (\'extra\', \'추가\'),\n                                              (\'junction\', \'정션\'), (\'split\', \'분리\')])\n        cap_detail = (\'<details><summary>오류별 점수 상한 보기</summary><table>\'\n                      + cap_rows + \'</table><p>복합 오류는 상한보다 더 낮아질 수 있습니다.</p></details>\')\n        if not result[\'priority_mode\']:\n            cap_detail = \'<p>이전 지수 곱셈 공식: 점수 상한 미적용</p>\'\n        cards.append(f\'<article class="indicator-card">{picture}\'\n                     f\'<div class="indicator-name">{number}. {html.escape(name)}</div>\'\n                     f\'<div class="indicator-score">{result["structural_score"]:.1f}<span> / 100</span></div>\'\n                     f\'<div class="indicator-note">{note if image is not None else "이미지 없는 수치 입력"}</div>\'\n                     f\'<div class="indicator-note">{extra_note}</div>{extra_detail}{cap_detail}\'\n                     \'<details><summary>획별 점수 보기</summary><table><thead><tr><th>획</th><th>점수</th>\'\n                     f\'<th>누락</th><th>분리</th><th>정션</th></tr></thead><tbody>{detail_rows}</tbody></table></details></article>\')\n    style = f\'\'\'<style>\n.indicator-gallery {{font-family:system-ui,sans-serif;color:#202124;}}\n.indicator-grid {{display:grid;grid-template-columns:repeat({columns},minmax(0,1fr));gap:16px;}}\n.indicator-card {{border:1px solid #ddd;border-radius:12px;padding:16px;background:white;min-width:0;}}\n.indicator-card img,.indicator-no-image {{width:100%;height:240px;object-fit:contain;background:white;}}\n.indicator-no-image {{display:flex;align-items:center;justify-content:center;color:#777;}}\n.indicator-name {{font-size:14px;overflow-wrap:anywhere;margin:10px 0 4px;}}\n.indicator-score {{font-size:36px;font-weight:700;}} .indicator-score span {{font-size:16px;font-weight:400;color:#666;}}\n.indicator-note {{font-size:12px;color:#666;margin:4px 0 12px;}}\n.indicator-gallery table {{font-size:12px;border-collapse:collapse;width:100%;margin-top:10px;}}\n.indicator-gallery td,.indicator-gallery th {{padding:5px;text-align:right;border-bottom:1px solid #eee;}}\n@media(max-width:650px){{.indicator-grid{{grid-template-columns:repeat(2,minmax(0,1fr));}}}}\n</style>\'\'\'\n    fragment = style + \'<section class="indicator-gallery"><p>사진별 구조 점수 · 누락·분리·정션·추가 잉크를 감점합니다.</p>\'\n    fragment += \'<div class="indicator-grid">\' + \'\'.join(cards) + \'</div></section>\'\n    gallery_path = destination/\'images_and_scores.html\'\n    gallery_path.write_text(\'<!doctype html><html lang="ko"><meta charset="utf-8">\'+fragment+\'</html>\', encoding=\'utf-8\')\n    for offset in range(0, len(loaded), per_page):\n        page = loaded[offset:offset+per_page]\n        cols = min(columns, len(page))\n        rows = math.ceil(len(page)/cols)\n        figure, axes = plt.subplots(rows, cols, squeeze=False, figsize=(3.5*cols, 4.2*rows), layout=\'constrained\')\n        for ax in axes.flat:\n            ax.axis(\'off\')\n        for ax, (image, name, result) in zip(axes.flat, page):\n            if image is not None:\n                ax.imshow(image)\n            else:\n                ax.text(.5, .5, \'Image unavailable\', transform=ax.transAxes, ha=\'center\', color=\'#777\')\n            safe_name = unicodedata.normalize(\'NFC\', name).encode(\'ascii\', \'backslashreplace\').decode(\'ascii\')\n            if len(safe_name) > 58:\n                safe_name = safe_name[:55]+\'...\'\n            ax.set_title(f\'{result["sample"]}. {safe_name}\\n{result["structural_score"]:.1f} / 100\', fontsize=12)\n        figure.suptitle(\'Image : structural score (0..100)\', fontsize=14)\n        path = destination/f\'images_and_scores_{offset//per_page+1:02d}.png\'\n        figure.savefig(path, dpi=150)\n        plt.close(figure)\n        pngs.append(str(path))\n    if show:\n        try:\n            from IPython import get_ipython\n            from IPython.display import HTML, display\n            if get_ipython() is not None:\n                display(HTML(fragment))\n        except ImportError:\n            pass\n    return str(gallery_path), pngs\n\n\ndef score_structure_report(report, config=None, save_dir=None, *, show_images=True,\n                           images_per_row=3, images_per_page=12, show_stroke_table=False):\n    """Score an existing structure_report.json/dict without rerunning extraction.\n\n    Returns an auditable report and saves glyph_scores.csv, stroke_scores.csv,\n    indicator_report.json. Uses M/D/J and remeasured extra ink; no filename-based scoring.\n    """\n    import pandas as pd\n\n    if any(not isinstance(v, int) or v < 1 for v in (images_per_row, images_per_page)):\n        raise ValueError(\'Gallery row/page counts must be positive integers\')\n    cfg = config or IndicatorConfig()\n    source_path = None\n    if isinstance(report, (str, Path)):\n        source_path = Path(report)\n        report = json.loads(source_path.read_text(encoding=\'utf-8-sig\'))\n    if not isinstance(report, dict) or not report.get(\'results\'):\n        raise ValueError(\'Provide structure_results or a structure_report.json with nonempty results\')\n    if save_dir is None:\n        base = source_path.parent if source_path else Path(report[\'results\'][0].get(\'npz\', \'.\')).parent\n        save_dir = base / (\'indicator_\' + datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\'))\n    destination = Path(save_dir)\n    destination.mkdir(parents=True, exist_ok=False)\n    scorer = StructuralIndicator(cfg)\n    glyph_rows, stroke_rows, results = [], [], []\n    for number, sample in enumerate(report[\'results\'], 1):\n        strokes = sample.get(\'strokes\', [])\n        if not strokes:\n            raise ValueError(f\'Sample {number} has no per-stroke measurements\')\n        missing, split, junction = [torch.tensor([s[key] for s in strokes], dtype=torch.float64)\n                                     for key in (\'missing\', \'split\', \'junction\')]\n        extra, extra_maps, extra_source = _extra_for_sample(sample, source_path, cfg)\n        with torch.no_grad():\n            output = scorer(missing, split, junction, extra=[extra])\n        identity = {\'sample\': number, \'csv_row\': sample.get(\'csv_row\'),\n                    \'input\': sample.get(\'input\') or sample.get(\'input_npz\') or \'\'}\n        glyph = {**identity, \'structural_score\': float(output[\'score\'][0]),\n                 \'priority_mode\': cfg.priority_mode,\n                 \'score_before_extra\': float(output[\'base_score\'][0]), \'extra\': extra,\n                 \'extra_retention\': float(output[\'extra_retention\'][0]),\n                 \'extra_assessed\': extra_source != \'disabled\',\n                 **{name+\'_score_cap\': float(value[0]) for name, value in output[\'caps\'].items()},\n                 \'mean_stroke_score\': float(output[\'mean_score\'][0]),\n                 \'worst_stroke_score\': float(output[\'worst_score\'][0]),\n                 \'stroke_count\': len(strokes)}\n        glyph_rows.append(glyph)\n        scored_strokes = []\n        for k, stroke in enumerate(strokes):\n            row = {**identity, \'stroke\': stroke.get(\'stroke\', k+1),\n                   \'missing\': float(missing[k]), \'split\': float(split[k]), \'junction\': float(junction[k]),\n                   \'stroke_score\': float(output[\'stroke_scores\'][0, k]),\n                   **{name+\'_retention\': float(value[0, k]) for name, value in output[\'retention\'].items()}}\n            stroke_rows.append(row)\n            scored_strokes.append(row)\n        extra_overlay = None\n        if extra_maps is not None:\n            import numpy as np\n            from PIL import Image\n            np.savez_compressed(destination/f\'extra_{number:04d}.npz\', **extra_maps)\n            background = np.repeat((1-extra_maps[\'image\'])[..., None], 3, axis=-1)\n            alpha = np.clip(extra_maps[\'extra_map\'], 0, 1)[..., None]\n            colored = background*(1-alpha) + np.array([1., 0., 0.])*alpha\n            extra_overlay = destination/f\'extra_{number:04d}.png\'\n            Image.fromarray(np.rint(colored*255).astype(np.uint8)).save(extra_overlay)\n        results.append({**glyph, \'extra_source\': extra_source,\n                        \'extra_overlay\': str(extra_overlay) if extra_overlay else None, \'strokes\': scored_strokes})\n    pd.DataFrame(glyph_rows).to_csv(destination / \'glyph_scores.csv\', index=False, encoding=\'utf-8-sig\')\n    pd.DataFrame(stroke_rows).to_csv(destination / \'stroke_scores.csv\', index=False, encoding=\'utf-8-sig\')\n    gallery, gallery_images = _score_gallery(report[\'results\'], results, destination, source_path,\n                                             images_per_row, images_per_page, show_images)\n    output_report = {\'indicator\': \'structural_indicator_v3_priority\', \'range\': [0, 100],\n                     \'uses_only\': [\'missing\', \'split\', \'junction\'] + ([\'extra\'] if cfg.extra_weight > 0 else []), \'trained\': False,\n                     \'calibrated_to_human_quality\': False, \'config\': asdict(cfg),\n                     \'measurement_config\': report.get(\'config\'), \'source_report\': str(source_path) if source_path else None,\n                     \'save_dir\': str(destination), \'gallery_html\': gallery,\n                     \'gallery_images\': gallery_images, \'results\': results}\n    (destination / \'indicator_report.json\').write_text(json.dumps(output_report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Structural indicator (0..100; higher is better; uncalibrated):\')\n    display = pd.DataFrame(glyph_rows)\n    display[\'input\'] = display[\'input\'].map(lambda p: str(p).replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1])\n    print(display.round(2).to_string(index=False))\n    if show_stroke_table:\n        print(\'\\nPer-stroke scores:\')\n        print(pd.DataFrame(stroke_rows)[[\'sample\', \'stroke\', \'missing\', \'split\', \'junction\', \'stroke_score\']]\n              .to_string(index=False, float_format=lambda x: f\'{x:.4g}\'))\n    print(\'Image : score gallery:\', gallery)\n    print(\'Saved:\', destination)\n    return output_report\n', _score_module.__file__, "exec"), _score_module.__dict__)


In [ ]:
from _structural_indicator_runtime import IndicatorConfig, score_structure_report
from pathlib import Path

if MODE == "values":
    indicator_source = {"results": [{"input": "manual_values", "strokes": VALUES, "extra": EXTRA_VALUE}]}
elif MODE == "auto":
    indicator_source = globals().get("structure_results")
    if indicator_source is None:
        if not REPORT_PATH:
            raise ValueError("구조 진단을 먼저 실행하거나 REPORT_PATH에 structure_report.json 경로를 넣으세요.")
        if str(REPORT_PATH).startswith("/content/drive/"):
            from google.colab import drive
            drive.mount("/content/drive")
        indicator_source = Path(REPORT_PATH)
else:
    raise ValueError('MODE는 "auto" 또는 "values"여야 합니다.')

indicator_results = score_structure_report(
    indicator_source, config=IndicatorConfig(**SCORE_CONFIG), save_dir=SAVE_DIR,
    show_images=SHOW_IMAGES, images_per_row=IMAGES_PER_ROW,
    images_per_page=IMAGES_PER_PAGE, show_stroke_table=SHOW_STROKE_TABLE,
)
